In [1]:
import polars as pl
import numpy as np
import plotly.graph_objects as go
import sqlite3

In [8]:
# Load data
# conn = sqlite3.connect("/Users/johng/Documents/JohnGrigoriadis_Thesis/hivemind_lamarckism/__data__/database.db")
# data = pl.read_database("SELECT * FROM individual", conn)
# # Load data
conn = sqlite3.connect("/Users/johng/Documents/PPSN_paper/PPSN_experiments/__data__/database.db")
data = pl.read_database("SELECT * FROM individual", conn)
data.tail()

id,alive,time_of_birth,time_of_death,requires_eval,fitness_,requires_init,genotype_,tags_
i64,i64,i64,i64,i64,f64,i64,str,str
2074,1,100,100,0,1.776215,0,"""[[-23.172701381320156, -12.233…","""{""mut"": true}"""
2075,1,100,100,0,1.963196,0,"""[[-11.56398831994185, 19.79010…","""{""mut"": true}"""
2076,1,100,100,0,1.989197,0,"""[[-4.359998451403193, 19.46086…","""{""mut"": true}"""
2077,1,100,100,0,1.840229,0,"""[[-18.687069128863232, 35.0641…","""{""mut"": true}"""
2078,1,100,100,0,1.755237,0,"""[[-25.529158622286303, 40.0940…","""{""mut"": true}"""


In [9]:
def get_fitness_per_df(data: pl.DataFrame, y_min: float | None = None) -> pl.DataFrame:
    min_gen = int(data["time_of_birth"].min())
    max_gen = int(data["time_of_birth"].max())

    generations = list(range(min_gen, max_gen + 1))

    id_to_fitness = dict(zip(data["id"].to_list(), data["fitness_"].to_list()))

    means, stds, bests, pop_sizes = [], [], [], []

    for gen in generations:
        
        mask = (
            (data["time_of_birth"] <= gen) &
            (data["time_of_death"] >= gen) &
            (data["requires_eval"] == 0)
        )


        ids = data.filter(mask)["id"].to_list()
        fits = np.array(
            [id_to_fitness[i] for i in ids if id_to_fitness.get(i) is not None],
            dtype=float
        )
        fits = fits[~np.isnan(fits)]

        pop_sizes.append(len(ids))

        if fits.size == 0:
            means.append(float("nan"))
            stds.append(float("nan"))
            bests.append(float("nan"))
        else:
            means.append(float(np.mean(fits)))
            stds.append(float(np.std(fits, ddof=0)))
            bests.append(float(np.min(fits)))

    pop_df = pl.DataFrame({
        "generation": generations,
        "pop_size": pop_sizes,
        "fitness_mean": means,
        "fitness_std": stds,
        "fitness_best": bests,
    })

    df = pop_df.filter(pl.col("fitness_mean").is_not_nan())

    x    = df["generation"].to_numpy()
    mean = df["fitness_mean"].to_numpy()
    std  = df["fitness_std"].to_numpy()
    best = df["fitness_best"].to_numpy()

    fig = go.Figure()

    # Std shading (add first so it renders behind the lines)
    fig.add_trace(go.Scatter(
        x=np.concatenate([x, x[::-1]]),
        y=np.concatenate([mean + std, (mean - std)[::-1]]),
        fill="toself",
        fillcolor="rgba(99, 110, 250, 0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        name="Mean ± Std",
    ))

    # Mean line
    fig.add_trace(go.Scatter(
        x=x, y=mean,
        mode="lines",
        name="Fitness mean",
        line=dict(color="rgba(99, 110, 250, 1.0)", width=2),
    ))

    # Best line
    fig.add_trace(go.Scatter(
        x=x, y=best,
        mode="lines",
        name="Fitness best",
        line=dict(color="rgba(239, 85, 59, 1.0)", width=2, dash="dash"),
    ))

    y_axis = dict(title="Fitness")
    if y_min is not None:
        y_axis["range"] = [y_min, 2]

    fig.update_layout(
        title=f"Fitness statistics per generation",
        xaxis=dict(title="Generation"),
        yaxis=y_axis,
        # legend=dict(x=0.01, y=0.99),
        hovermode="x unified",
        # template="plotly_dark",  # remove or change if you prefer light mode
        width=900,
        height=500,
    )

    fig.show()
    # return df

In [10]:
get_fitness_per_df(data, y_min=0)

In [13]:
print(len(data.filter(pl.col("fitness_") == 2.0)))

0


In [14]:
import json

def parse_genotype(s: str) -> np.ndarray | None:
    """Parse a genotype string into a flat numpy array."""
    try:
        arr = np.array(json.loads(s), dtype=float)
        return arr.flatten()
    except Exception:
        try:
            arr = np.array(eval(s), dtype=float)
            return arr.flatten()
        except Exception:
            return None


def get_diversity_per_gen(data: pl.DataFrame) -> None:
    """
    Plot population diversity (mean pairwise distance) over generations.
    Individuals alive at generation g are those with time_of_birth <= g <= time_of_death
    and requires_eval == 0.
    """
    min_gen = int(data["time_of_birth"].min())
    max_gen = int(data["time_of_birth"].max())
    generations = list(range(min_gen, max_gen + 1))

    # Pre-parse all genotypes
    ids = data["id"].to_list()
    geno_strs = data["genotype_"].to_list()
    id_to_geno = {i: parse_genotype(g) for i, g in zip(ids, geno_strs)}

    diversities = []
    valid_gens = []

    for gen in generations:
        mask = (
            (data["time_of_birth"] <= gen) &
            (data["time_of_death"] >= gen) &
            (data["requires_eval"] == 0)
        )
        alive_ids = data.filter(mask)["id"].to_list()
        genos = [id_to_geno[i] for i in alive_ids if id_to_geno.get(i) is not None]
        genos = [g for g in genos if g is not None]

        if len(genos) < 2:
            continue

        G = np.stack(genos)  # shape (n, d)
        # Mean distance to centroid — O(n) and equivalent to half mean pairwise dist
        centroid = G.mean(axis=0)
        diversity = float(np.mean(np.linalg.norm(G - centroid, axis=1)))
        diversities.append(diversity)
        valid_gens.append(gen)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=valid_gens,
        y=diversities,
        mode="lines",
        name="Diversity (mean dist to centroid)",
        line=dict(color="rgba(0, 204, 150, 1.0)", width=2),
    ))
    fig.update_layout(
        title="Population diversity per generation",
        xaxis=dict(title="Generation"),
        yaxis=dict(title="Mean distance to centroid"),
        hovermode="x unified",
        width=900,
        height=500,
    )
    fig.show()

In [15]:
get_diversity_per_gen(data)